# Bài tập Buổi 5 — Pipeline Machine Learning: EDA & Tiền xử lý trên Titanic

**Khóa học hè 2026 — Python & Machine Learning · ML IoT Lab, HCMUT**

---

## Bối cảnh

Bạn vừa nhận được dataset **Titanic** — danh sách hành khách và việc họ có sống sót sau thảm họa hay không.
Nhiệm vụ của bạn **không phải** huấn luyện mô hình, mà là hoàn thành **hai bước đầu và quan trọng nhất** của một dự án Machine Learning:

> **Khám phá dữ liệu (EDA) → Tiền xử lý dữ liệu.**

Đây là phần chiếm ~70–90% công sức thực tế của một dự án ML. Một mô hình mạnh không thể cứu một bộ dữ liệu kém chất lượng.

## Mục tiêu bài tập

Sau khi hoàn thành, bạn sẽ chứng minh được rằng mình có thể:

1. Thực hiện **EDA đầy đủ** trên một dataset thực tế: kiểm tra cấu trúc, giá trị thiếu, outlier, phân phối và tương quan.
2. **Trực quan hóa** dữ liệu và **rút ra nhận xét có căn cứ** (không chỉ vẽ hình cho đẹp).
3. Áp dụng **đúng kỹ thuật tiền xử lý** cho từng loại dữ liệu: xử lý missing, encoding, scaling.
4. Chia tập và xây pipeline tiền xử lý **không rò rỉ dữ liệu (data leakage)**.
5. Viết **nhận xét tổng hợp** về dữ liệu như một data analyst thực thụ.

## Yêu cầu nộp bài

- Hoàn thiện notebook này (điền vào tất cả các ô `# TODO` và các phần *"Trả lời:"*).
- Notebook phải **chạy được từ trên xuống dưới không lỗi** (Kernel → Restart & Run All).
- Nộp qua **GitHub**: tải repo mẫu → đưa lên repo cá nhân → làm bài và nộp trên đó.

## Tiêu chí chấm (10 điểm)

| Nội dung | Điểm |
|---|---|
| EDA đầy đủ (shape/info/missing/outlier) | 2.0 |
| Trực quan hóa + nhận xét cho mỗi biểu đồ | 2.0 |
| Xử lý missing & outlier hợp lý, có giải thích | 1.5 |
| Encoding & scaling đúng loại biến | 1.5 |
| Chia tập & tiền xử lý **không leakage** | 1.5 |
| Nhận xét tổng hợp về dữ liệu | 1.5 |

> **Lưu ý về liêm chính học thuật:** được tham khảo tài liệu, nhưng phải **tự viết code và tự hiểu**. Phần nhận xét phải là quan sát của chính bạn từ dữ liệu.

---


## 0. Chuẩn bị môi trường

Ô này đã viết sẵn — chạy để nạp thư viện. Nếu thiếu thư viện, cài bằng `pip install pandas numpy matplotlib seaborn scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)          # cố định ngẫu nhiên -> kết quả tái lập được
print("Sẵn sàng.")

## 1. Tải dữ liệu (đã cho)

Ô này đã viết sẵn. Dữ liệu được tải từ `seaborn`, có fallback tải từ Internet nếu cần.

In [ ]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

---
## Task 1 — Loại bỏ cột rò rỉ nhãn (data leakage) và cột dư thừa

### Mục đích
Trên slide đã học: **data leakage** là khi thông tin không được phép "rò" vào mô hình, khiến kết quả đẹp trên giấy nhưng vô dụng thực tế. Bản Titanic của seaborn chứa sẵn nhiều cột **rò rỉ nhãn** hoặc **trùng lặp**:

- `alive` (yes/no) — chính là `survived` viết bằng chữ ⇒ **rò rỉ target trực tiếp**. Để lại là mô hình "gian lận".
- `who`, `adult_male`, `class` — được suy ra từ `sex`, `age`, `pclass` (trùng thông tin).
- `deck` — thiếu quá nhiều (~77%).
- `embark_town` — trùng `embarked`; `alone` — suy ra từ `sibsp` + `parch`.

### Yêu cầu
1. In ra danh sách cột và tỷ lệ missing của **toàn bộ** dataframe (để thấy `deck` thiếu bao nhiêu).
2. Loại bỏ các cột rò rỉ / dư thừa ở trên, chỉ giữ lại:
   `survived, pclass, sex, age, sibsp, parch, fare, embarked`.
3. **Trả lời** (markdown ô dưới): vì sao để lại cột `alive` sẽ khiến mô hình đạt accuracy ~100% mà không thực sự học được gì?

### Gợi ý
- `df.isnull().mean()` cho tỷ lệ thiếu theo cột.
- `df.drop(columns=[...])` để bỏ cột.

In [ ]:
# TODO 1a: in tỷ lệ missing của tất cả các cột
print("Tỷ lệ missing của từng cột:")
print(df.isnull().mean() * 100)

# TODO 1b: bỏ các cột rò rỉ/dư thừa, gán lại vào biến df
leaky = ['alive', 'who', 'adult_male', 'class', 'deck', 'embark_town', 'alone']     # điền danh sách cột cần bỏ (chỉ những cột có trong df)
df = df.drop(columns=leaky)

print("Các cột còn lại:", list(df.columns))

**Trả lời 1c (vì sao `alive` gây rò rỉ target):**

Cột `alive` mang thông tin giống hệt biến mục tiêu `survived` (chỉ khác định dạng là 'yes'/'no' thay vì 1/0). Nếu giữ lại, mô hình sẽ trực tiếp "nhìn đáp án" để dự đoán thay vì học cách suy luận từ các đặc trưng khác (tuổi, giới tính, hạng vé...). Điều này dẫn đến data leakage, làm mô hình đạt độ chính xác gần 100% lúc huấn luyện nhưng hoàn toàn vô dụng khi dự đoán dữ liệu mới trong thực tế.

---
## Task 2 — Quan sát tổng quan

### Mục đích
Trước khi phân tích sâu, phải nắm được "hình dạng" của dữ liệu: bao nhiêu mẫu, bao nhiêu đặc trưng, kiểu dữ liệu từng cột, và thống kê cơ bản. Đây là bước đầu tiên của mọi EDA.

### Yêu cầu
1. In **số dòng và số cột**; nêu rõ đâu là **biến mục tiêu (target)**.
2. Dùng `df.info()` để xem kiểu dữ liệu và số giá trị non-null.
3. Dùng `df.describe()` cho biến số và `df.describe(include="object")` (hoặc `"category"`) cho biến phân loại.
4. **Trả lời:** cột nào là biến **số**, cột nào là biến **phân loại**?

In [ ]:
# TODO 2: shape, info, describe
print(f"Số dòng: {df.shape[0]}, Số cột: {df.shape[1]}")
print("Biến mục tiêu (target) là: survived\n")

print("--- THÔNG TIN KIỂU DỮ LIỆU ---")
df.info()

print("\n--- THỐNG KÊ CƠ BẢN (BIẾN SỐ) ---")
display(df.describe())

print("\n--- THỐNG KÊ CƠ BẢN (BIẾN PHÂN LOẠI) ---")
display(df.describe(include=["object", "category"]))

**Trả lời 2 (biến số vs biến phân loại):**

- **Biến số (Numerical):** `age` (tuổi), `fare` (giá vé), `sibsp` (số anh chị em/vợ chồng), `parch` (số cha mẹ/con cái).
- **Biến phân loại (Categorical):** `survived` (sống sót - phân loại nhị phân), `pclass` (hạng vé - phân loại có thứ tự), `sex` (giới tính), `embarked` (cảng lên tàu).

---
## Task 3 — Missing Value: thống kê & đề xuất cách xử lý

### Mục đích
Mô hình học máy **không nhận trực tiếp giá trị NaN**. Nhưng cách xử lý phụ thuộc **tỷ lệ thiếu** và **vai trò của cột** — không có một cách đúng cho mọi trường hợp.

### Yêu cầu
1. Lập bảng: mỗi cột còn missing → **số lượng** và **phần trăm** thiếu.
2. Với **từng cột** còn thiếu, **đề xuất** cách xử lý và **giải thích ngắn gọn** (xóa / điền mean / điền median / điền mode / KNN...).

### Gợi ý
- Nhắc lại từ slide: `median` bền vững hơn `mean` khi có outlier; cột thiếu quá nhiều (>~60–70%) thường nên **bỏ**; biến phân loại thường điền **mode**.

In [ ]:
# TODO 3: bảng missing (count + %)
missing_table = pd.DataFrame({
    'Số lượng thiếu': df.isnull().sum(),
    'Phần trăm thiếu (%)': (df.isnull().mean() * 100).round(2)
})
missing_table = missing_table[missing_table['Số lượng thiếu'] > 0]
display(missing_table)

**Trả lời 3 (đề xuất xử lý cho từng cột thiếu):**

| Cột | % thiếu | Cách xử lý đề xuất | Lý do |
|---|---|---|---|
| `age` | ~19.87% | Điền giá trị trung vị (Median Imputation) | Tỷ lệ thiếu vừa phải. Cột `age` có thể bị ảnh hưởng bởi các hành khách rất già (outlier), nên dùng median sẽ bền vững hơn mean. |
| `embarked` | ~0.22% | Điền giá trị xuất hiện nhiều nhất (Mode Imputation) | Là biến phân loại và tỷ lệ thiếu cực kỳ nhỏ (chỉ 2 dòng), điền mode là cách đơn giản và không làm thay đổi phân phối. |

---
## Task 4 — Phát hiện Outlier & **ra quyết định**

### Mục đích
Outlier có thể là **lỗi nhập liệu** (cần xử lý) hoặc **hiện tượng thật** (cần giữ). Phát hiện thôi chưa đủ — một analyst giỏi phải **quyết định** làm gì và giải thích được.

### Yêu cầu
1. Trên hai cột `age` và `fare`, đếm số outlier bằng **cả hai** phương pháp: **IQR** và **Z-score** (ngưỡng |z| > 3).
2. **Trả lời:** với các outlier của `fare`, bạn **giữ lại hay loại bỏ**? Vì sao? (gợi ý: nghĩ xem vé đắt bất thường là lỗi hay là vé hạng nhất có thật).

### Gợi ý
- IQR: outlier là điểm ngoài khoảng `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.
- Z-score: `from scipy import stats; np.abs(stats.zscore(series.dropna()))`.

In [ ]:
# TODO 4: đếm outlier theo IQR và Z-score cho 'age' và 'fare'
def dem_outlier_iqr(s):
    s_clean = s.dropna()
    Q1 = s_clean.quantile(0.25)
    Q3 = s_clean.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return ((s_clean < lower_bound) | (s_clean > upper_bound)).sum()

def dem_outlier_zscore(s, nguong=3.0):
    s_clean = s.dropna()
    z_scores = np.abs(stats.zscore(s_clean))
    return (z_scores > nguong).sum()

for col in ["age", "fare"]:
    print(f"Cột '{col}':")
    print(f" - Số outlier (IQR): {dem_outlier_iqr(df[col])}")
    print(f" - Số outlier (Z-score): {dem_outlier_zscore(df[col])}")

**Trả lời 4 (quyết định với outlier của `fare`):**

**Giữ lại các outlier của `fare`.** Vì vé đắt bất thường không phải là lỗi nhập liệu (data entry error) mà phản ánh đúng thực tế lịch sử: có những hành khách siêu giàu mua vé hạng nhất (First Class) với giá cực cao. Đặc trưng này chứa tín hiệu (signal) quan trọng, vì những người trả giá vé cao thường được ưu tiên lên thuyền cứu sinh, ảnh hưởng trực tiếp đến khả năng sống sót. Thay vì loại bỏ, ta sẽ dùng thuật toán chuẩn hóa kháng nhiễu (RobustScaler) ở phần tiền xử lý.

---
## Task 5 — Trực quan hóa & nhận xét

### Mục đích
EDA là môn học về **nhìn** dữ liệu. Mỗi biểu đồ phải trả lời một câu hỏi và **đi kèm một nhận xét**. Vẽ mà không nhận xét thì không tính điểm.

### Yêu cầu — vẽ tối thiểu 4 loại biểu đồ, mỗi biểu đồ 1–2 câu nhận xét:
1. **Univariate — Histogram**: phân phối của `age` và `fare`. (Nhận xét: có lệch không? lệch trái hay phải?)
2. **Univariate — Boxplot**: `fare` theo nhóm `survived` hoặc `pclass`. (Nhận xét: outlier, trung vị.)
3. **Bivariate — Bar/Barplot**: **tỷ lệ sống sót** theo `sex` và theo `pclass`. (Nhận xét: nhóm nào sống nhiều hơn, chênh bao nhiêu %?)
4. **Multivariate — Heatmap**: ma trận tương quan giữa các biến số. (Nhận xét: cặp biến nào tương quan mạnh?)

### Gợi ý
- `sns.histplot`, `sns.boxplot`, `sns.barplot(data=df, x="sex", y="survived")` (barplot tự tính trung bình = tỷ lệ sống sót), `sns.heatmap(df.select_dtypes("number").corr(), annot=True)`.

In [ ]:
# TODO 5a: Histogram age & fare
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(df['age'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Phân phối Tuổi (Age)')

sns.histplot(df['fare'], kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Phân phối Giá vé (Fare)')
plt.tight_layout()
plt.show()

In [ ]:
# TODO 5b: Boxplot fare theo survived hoặc pclass
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='pclass', y='fare', palette='Set2')
plt.title('Phân bố Giá vé (Fare) theo Hạng vé (Pclass)')
plt.show()

In [ ]:
# TODO 5c: Barplot tỷ lệ sống sót theo sex và pclass
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=df, x='sex', y='survived', ax=axes[0], palette='pastel')
axes[0].set_title('Tỷ lệ sống sót theo Giới tính')

sns.barplot(data=df, x='pclass', y='survived', ax=axes[1], palette='pastel')
axes[1].set_title('Tỷ lệ sống sót theo Hạng vé')
plt.tight_layout()
plt.show()

In [ ]:
# TODO 5d: Heatmap correlation
plt.figure(figsize=(8, 6))
corr_matrix = df.select_dtypes(include='number').corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Ma trận tương quan giữa các biến số')
plt.show()

**Nhận xét 5 (viết cho từng biểu đồ ở trên):**

- **Histogram**: `age` có phân phối gần chuẩn nhưng hơi lệch phải (tập trung đông ở độ tuổi 20-30). `fare` bị lệch phải cực kỳ nặng (right-skewed), đại đa số là vé rẻ nhưng có một cái đuôi kéo dài bởi các vé siêu đắt.
- **Boxplot**: Hạng vé 1 có mức giá (trung vị) cao nhất và biên độ dao động cực rộng với nhiều outlier khổng lồ. Hạng 2 và 3 có giá vé thấp, hội tụ hẹp.
- **Bar survival**: Phụ nữ có tỷ lệ sống sót vượt trội (~74%) so với đàn ông (~18%). Hành khách Hạng 1 sống sót nhiều nhất (>60%), và giảm dần xuống Hạng 3 (~24%).
- **Heatmap**: `pclass` có tương quan nghịch mạnh với `fare` (-0.55), hợp lý vì hạng vé số càng nhỏ (Hạng 1) thì giá càng cao. `sibsp` và `parch` có tương quan thuận khá cao (0.41), thể hiện việc người ta đi theo cụm gia đình.

---
## Task 6 — Chia tập **TRƯỚC** khi tiền xử lý (chống data leakage)

### Mục đích
Đây là điểm mấu chốt của buổi học. Mọi phép "học tham số" từ dữ liệu (median để điền, min/max/IQR để scale, danh mục để encode) **chỉ được học từ tập train**. Nếu học từ toàn bộ dữ liệu rồi mới chia, thông tin của tập test đã **rò rỉ** — điểm đánh giá sẽ ảo.

⇒ **Vì vậy phải chia tập TRƯỚC**, rồi mới xử lý.

### Yêu cầu
1. Tách `X` (đặc trưng) và `y` (`survived`).
2. Chia **train / validation / test** theo tỷ lệ khoảng **70 / 15 / 15**, có **`stratify=y`** để giữ nguyên tỷ lệ hai lớp.
3. In shape của 3 tập và **tỷ lệ sống sót** trong mỗi tập (để kiểm tra stratify hoạt động).

### Gợi ý
- Dùng `train_test_split` **hai lần**: lần 1 tách test (15%), lần 2 tách val từ phần còn lại.
- `stratify` nhận vào nhãn tương ứng ở mỗi lần chia.

In [ ]:
# TODO 6: chia train/val/test có stratify
X = df.drop(columns=['survived'])
y = df['survived']

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

print(f"Shape Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Tỷ lệ Survived - Train: {y_train.mean():.2f} | Val: {y_val.mean():.2f} | Test: {y_test.mean():.2f}")

---
## Task 7 — Xây pipeline tiền xử lý, **fit chỉ trên train**

### Mục đích
Gộp toàn bộ bước tiền xử lý vào một `ColumnTransformer` + `Pipeline`, `fit` **một lần trên `X_train`** rồi `transform` cho val/test. Đây là cách chuẩn để **đảm bảo không leakage** và tái sử dụng được.

### Yêu cầu
Xây `preprocess` gồm:

- **Biến số** (`age`, `sibsp`, `parch`, `fare`): `SimpleImputer(median)` → scaler (chọn `RobustScaler` vì `fare` có outlier, hoặc giải thích lựa chọn khác).
- **Biến phân loại** (`sex`, `embarked`): `SimpleImputer(most_frequent)` → `OneHotEncoder`.
- **Biến thứ tự** (`pclass`): giữ nguyên (`passthrough`) vì đã là số có thứ tự 1 < 2 < 3.

Sau đó: `fit` trên `X_train`, `transform` cho cả ba tập; in shape kết quả và tên cột sau biến đổi.

### Yêu cầu trả lời
- **Trả lời:** giải thích vì sao `fit` chỉ trên train (không phải trên toàn bộ dữ liệu) thì tránh được leakage.

### Gợi ý
- Khung `ColumnTransformer([... ("num", pipe_so, num_cols), ("cat", pipe_cat, cat_cols), ("ord", "passthrough", ord_cols)])`.
- `preprocess.get_feature_names_out()` để xem tên cột sau biến đổi.

In [ ]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

# TODO 7: xây pipeline cho biến số và biến phân loại
pipe_so = Pipeline([
    ("imputer", SimpleImputer(strategy='median')),
    ("scaler", RobustScaler())  # Dùng RobustScaler để chống nhiễu từ outlier của fare
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("onehot", OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocess = ColumnTransformer([
    ("num", pipe_so, num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols)
])

preprocess.fit(X_train)               # fit CHỈ trên train
X_train_t = preprocess.transform(X_train)
X_val_t = preprocess.transform(X_val)
X_test_t = preprocess.transform(X_test)

print("Kích thước X_train sau biến đổi:", X_train_t.shape)
print("Danh sách các cột:", list(preprocess.get_feature_names_out()))

**Trả lời 7 (vì sao fit chỉ trên train tránh leakage):**

Nếu ta `fit` (tính toán median, tỷ lệ phân phối, giá trị encode...) trên toàn bộ dữ liệu trước khi chia tập, các đặc trưng thống kê của tập Test/Validation đã bị mô hình "nhìn thấy" và trộn lẫn vào tập Train. Phải `fit` duy nhất trên tập Train, sau đó dùng các tham số đã học được này để `transform` cho Val/Test, như vậy mới mô phỏng đúng bối cảnh thực tế: mô hình gặp dữ liệu hoàn toàn mới và chưa từng biết trước thông tin thống kê của chúng.

---
## Task 8 — Câu hỏi tư duy: chọn metric đánh giá

### Mục đích
Buổi học nhấn mạnh: **không có metric tốt nhất tuyệt đối** — phải chọn theo bài toán và mức mất cân bằng dữ liệu. Bài này không cần code, chỉ cần lập luận.

### Yêu cầu — trả lời ngắn gọn:
1. Biến mục tiêu `survived` có **mất cân bằng** không? (tính tỷ lệ hai lớp để trả lời).
2. Nếu chỉ nhìn **Accuracy**, có thể bị đánh lừa trong trường hợp nào?
3. Với bài toán Titanic, bạn sẽ ưu tiên metric nào (Accuracy / Precision / Recall / F1)? Vì sao?

In [ ]:
# TODO 8: tính tỷ lệ hai lớp của 'survived' để hỗ trợ trả lời
print("Tỷ lệ các lớp trong target 'survived':")
print(df['survived'].value_counts(normalize=True).round(2))

**Trả lời 8:**

1. Biến mục tiêu có mất cân bằng nhẹ. Có khoảng 62% hành khách thiệt mạng (0) và 38% sống sót (1).
2. Nếu tập dữ liệu mất cân bằng nặng (ví dụ 99% chết, 1% sống), một mô hình đoán bừa 100% "chết" sẽ đạt Accuracy 99%, nhưng thực chất nó không tìm ra được ai sống sót cả. Accuracy bị ảo và đánh lừa ta về độ hiệu quả của mô hình.
3. Tôi ưu tiên dùng **F1-score** (hoặc Recall nếu chi phí bỏ lót người sống sót cao). F1-score là trung bình điều hòa giữa Precision và Recall, giúp đánh giá công bằng hơn trên tập dữ liệu mất cân bằng, đảm bảo mô hình thực sự phân loại tốt cả 2 lớp chứ không phải đoán hùa theo lớp đa số.

---
## Task 9 — Nhận xét tổng hợp về dữ liệu

### Mục đích
Khép lại toàn bộ EDA bằng một bản tóm tắt như một data analyst gửi cho đồng đội: **những gì đáng chú ý nhất** về bộ dữ liệu này.

### Yêu cầu — viết ít nhất 5 gạch đầu dòng, dựa trên **bằng chứng** (số liệu / biểu đồ) ở trên:
- Đặc trưng nào **tương quan mạnh nhất** với khả năng sống sót? (số liệu chứng minh)
- Cột nào **thiếu nhiều nhất** và bạn đã xử lý thế nào?
- Biến mục tiêu có **mất cân bằng** không? ảnh hưởng gì tới việc chọn metric?
- Đặc trưng nào cần **scaling**, đặc trưng nào cần **encoding**? vì sao?
- Một điều bạn thấy **bất ngờ / thú vị** trong dữ liệu.

**Nhận xét tổng hợp của bạn:**

1. **Yếu tố quyết định sinh tử:** Giới tính (`sex`) và hạng vé (`pclass`) có ảnh hưởng mạnh nhất. Phụ nữ (~74%) và hành khách Hạng 1 (>60%) có tỷ lệ sống sót cao vượt trội so với nam giới và Hạng 3.
2. **Xử lý thiếu hụt:** Cột `age` thiếu ~20% dữ liệu, được điền bằng Median để tránh lệch do outlier tuổi tác. Cột `embarked` thiếu rất ít, được điền bằng Mode. Những cột thiếu nghiêm trọng hoặc rò rỉ dữ liệu như `deck`, `alive` đã bị xóa.
3. **Mất cân bằng nhãn:** Dữ liệu có sự chênh lệch (62% tử vong vs 38% sống sót). Sự mất cân bằng này đòi hỏi ta phải chú ý sử dụng F1-score hoặc Precision/Recall để đánh giá mô hình thay vì chỉ dùng Accuracy.
4. **Quyết định Scale và Encode:** Biến phân loại `sex`, `embarked` được OneHot Encoding để thuật toán học được. Các biến số, đặc biệt là `fare`, chứa nhiều outlier thật (giá vé hạng nhất), nên được xử lý bằng `RobustScaler` (dựa trên phân vị) để không làm sai lệch phân phối tổng thể.
5. **Điểm thú vị (Outlier mang tín hiệu):** Ban đầu, các giá trị giá vé cực cao trông giống như lỗi dữ liệu (outliers). Nhưng qua quá trình EDA, ta nhận thấy chúng phản ánh đúng hành khách giới tinh hoa trên tàu Titanic. Đặc trưng này hoàn toàn hợp lệ và có sức mạnh phân loại rất tốt (những người mua vé đắt có tỷ lệ được cứu cao hơn).

---
## (Bonus — không bắt buộc) Thử thách nâng cao

Chọn **một** trong các hướng sau nếu bạn muốn thử sức:

1. **Feature engineering:** tạo đặc trưng mới `family_size = sibsp + parch + 1`, hoặc trích `title` (Mr/Mrs/Miss...) từ tên (nếu dùng bản có cột `name`). Kiểm tra tương quan với `survived`.
2. **So sánh scaler:** vẽ phân phối `fare` trước và sau khi áp `StandardScaler`, `MinMaxScaler`, `RobustScaler`. Nhận xét scaler nào phù hợp nhất với dữ liệu lệch + có outlier.
3. **Bẫy KNN:** thử `KNNImputer` để điền `age` **khi chưa scale** và **sau khi đã scale** `fare`. Quan sát kết quả có khác nhau không, và giải thích tại sao (gợi ý: khoảng cách Euclid bị chi phối bởi cột thang đo lớn).

In [ ]:
# (tùy chọn) code cho phần Bonus
...

---
## Bảng tự kiểm trước khi nộp

- [ ] Notebook chạy **Restart & Run All** không lỗi.
- [ ] Đã bỏ các cột rò rỉ/dư thừa (Task 1) và giải thích được vì sao.
- [ ] Mỗi biểu đồ (Task 5) đều có **nhận xét**.
- [ ] Đã **chia tập trước**, tiền xử lý **fit chỉ trên train** (Task 6–7).
- [ ] Đã trả lời tất cả các phần *"Trả lời:"*.
- [ ] Nhận xét tổng hợp (Task 9) có **ít nhất 5 ý** dựa trên bằng chứng.
- [ ] Đã push lên **repo cá nhân trên GitHub**.
